In [39]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
sns.set_context("notebook", font_scale=2)

# Copy files in separate folders
rsync -v harvines@wynton:~/lab/explain_score_test/src/simulation_comparators/_output/dim4_linear_std/accuracy/0.05/40/8000/resultestimate1.csv .
rsync -v harvines@wynton:~/lab/explain_score_test/src/simulation_comparators/_output/dim4_norm2_std/accuracy/0.02/40/8000/resultestimate1.csv .
rsync -v harvines@wynton:~/lab/explain_score_test/src/casestudy/_output/acs_pubcov_save_scaled/MLPClassifier/accuracy/0.05/40/resultestimate1.csv .
rsync -v harvines@wynton:~/lab/explain_score_test/src/casestudy/_output/acs_pubcov_save_scaled/MLPClassifier/accuracy/0.05/40/comp_resultestimate1.csv .
rsync -v harvines@wynton:~/lab/explain_score_test/src/casestudy/_output/acs_pubcov_covariate_scaled/MLPClassifier/accuracy/0.05/40/resultestimate1.csv .
rsync -v harvines@wynton:~/lab/explain_score_test/src/casestudy/_output/acs_pubcov_covariate_scaled/MLPClassifier/accuracy/0.05/40/comp_resultestimate1.csv .
rsync -rv harvines@wynton:~/lab/explain_score_test/src/casestudy/_output/readmission_ucsf_zsfg_flip_ptrim_outcome/GradientBoostingClassifier/accuracy/0.02/40/resultestimate1.csv .
rsync -rv harvines@wynton:~/lab/explain_score_test/src/casestudy/_output/readmission_ucsf_zsfg_flip_ptrim_outcome/GradientBoostingClassifier/accuracy/0.05/40/comp_resultestimate1.csv .
rsync -rv harvines@wynton:~/lab/explain_score_test/src/casestudy/_output/readmission_ucsf_zsfg_flop_filter_trim_ptrim_calibrate/GradientBoostingClassifier/accuracy/0.02/40/resultestimate1.csv
rsync -rv harvines@wynton:~/lab/explain_score_test/src/casestudy/_output/readmission_ucsf_zsfg_flop_filter_trim_ptrim_calibrate/GradientBoostingClassifier/accuracy/0.02/40/comp_resultestimate1.csv

In [40]:
SIGNIFICANCE_LEVEL = 0.05
BONFERRONI_CORRECTION = False  # for comparators
COMPARATOR_NAMES = {
    "Cond_Cov": "Covariate",
    "Cond_Outcome": "Outcome",
    "KCICovariateTest": "KCI $\dagger$",
    "ParametricChangeExplanation": "ParamY $\ddag$",
    "ParametricAccExplanation": "ParamLoss $\ddag$",
    "KCIOutcomeTestAgg": "KCI",
    "KCIOutcomeTest": "KCI $\dagger$",
    "TEVIMTest": "TE-VIM $\dagger$",
    "KSTest": "KS $\ddag$",
    # "DomainClassifierExplanation",
    "ScoreMethod": "Score $\ddag$",
    "KCICovariateTestAgg": "KCI",
    # "LinearMediationTest",
}

def parse_proposed_vars(var_str):
    mask = np.array(var_str.replace('(','').replace(')','').replace(' ','').split(',')) == 'False'
    mask_false_idx = np.where(mask)[0]
    mask_false = ["X%d" % (i+1) for i in mask_false_idx]
    return ",".join(mask_false)

def parse_comp_vars(var_str, num_p):
    all_vars = ["X%d" % (i+1) for i in np.arange(num_p)]  # 1-indexed
    excluded_indices = var_str.split(",")
    included_indices = [i for i in all_vars if i not in excluded_indices]
    return ",".join(included_indices)

cmap = LinearSegmentedColormap.from_list('custom_cmap', [(0,'Red'),(1,'Grey')])

In [41]:
SIMULATION_SETTINGS = {
    'shield_nonzero_covariate': {
        'grouping_data': '/Volumes/jhong9_SHIELD_shared/for_jean/model_data/shield_feature_groupings_drop_missing_nonzero_discretize.csv',
        'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/shield_nonzero_covariate/resultestimate1.csv',
        'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/acs_pubcov_scaled_outcome/comp_resultestimate1.csv',
        'target_data': "/Volumes/jhong9_SHIELD_shared/for_jean/model_data/ucsf_drop_missing_nonzero_discretize.csv",
        'decomp': "Cond_Cov",
    },
    'acs_pubcov_outcome': {
        'grouping_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/acs_pubcov_feature_groupings_groups.csv',
        'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/acs_pubcov_scaled_outcome/resultestimate1.csv',
        'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/acs_pubcov_scaled_outcome/comp_resultestimate1.csv',
        'target_data': "/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/acs_pubcov_target.csv",
        'decomp': "Cond_Outcome",
    }, 
    'acs_pubcov_covariate': {
        'grouping_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/acs_pubcov_feature_groupings_groups.csv',
        'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/acs_pubcov_scaled_covariate/resultestimate1.csv',
        'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/acs_pubcov_scaled_covariate/comp_resultestimate1.csv',
        'target_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/acs_pubcov_target.csv',
        'decomp': "Cond_Cov",
    },
    'readmission_ucsf_zsfg_covariate': {
        'grouping_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_perf_changes/data/hf_feature_groupings_groups_filter_trim.csv',
        'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/readmission_ucsf_zsfg_covariate/resultestimate1.csv',
        # 'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/readmission_ucsf_zsfg_covariate/resultestimate1_ucsf_zsfg_readmit_covariate_zero.csv',
        'target_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_perf_changes/data/zsfg_data_filter_trim.csv',
        'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/readmission_ucsf_zsfg_covariate/comp_resultestimate1.csv',
        'decomp': "Cond_Cov",
    },
    'readmission_ucsf_zsfg_outcome': {
        'grouping_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_perf_changes/data/hf_feature_groupings_groups_filter_trim.csv',
        'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/readmission_ucsf_zsfg_outcome/resultestimate1.csv',
        'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/readmission_ucsf_zsfg_outcome/comp_resultestimate1.csv',
        'target_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_perf_changes/data/zsfg_data_filter_trim.csv',
        'decomp': "Cond_Outcome",
    },
    'readmission_zsfg_ucsf_fixed_covariate': {
        'grouping_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/readmission/zsfg_ucsf_fixed_encounters/hf_feature_groupings_groups_filter_trim.csv',
        'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/casestudy/_output/readmission_zsfg_ucsf_fixed_encounters/GradientBoostingClassifier/accuracy/0.02/0.05/40/Cond_Cov/resultestimate1.csv',
        'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/casestudy/_output/readmission_zsfg_ucsf_fixed_encounters/GradientBoostingClassifier/accuracy/0.02/0.05/40/Cond_Cov/comp_resultestimate1.csv',
        'target_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/readmission/zsfg_ucsf_fixed_encounters/ucsf_test_fixed_encounters_imputed_trim.csv',
        'decomp': "Cond_Cov",
    },
    'readmission_zsfg_ucsf_prior_covariate': {
        'grouping_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/readmission/zsfg_ucsf_fixed_encounters/hf_feature_groupings_groups_filter_trim.csv',
        'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/casestudy/_output/readmission_zsfg_ucsf_prior_encounters/GradientBoostingClassifier/accuracy/0.02/0.05/40/Cond_Cov/resultestimate1.csv',
        'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/casestudy/_output/readmission_zsfg_ucsf_prior_encounters/GradientBoostingClassifier/accuracy/0.02/0.05/40/Cond_Cov/comp_resultestimate1.csv',
        'target_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/data/readmission/ucsf_test_prior_encounters_imputed_trim.csv',
        'decomp': "Cond_Cov",
    }
}

In [31]:
def plot_aggregate(result_data):
    df = pd.read_csv(result_data)
    df_agg = df[(df.decomp == 'agg') & (df.est == 'onestep')]
    df_agg.loc[:,'vars'] = df_agg.loc[:, 'vars'].replace('X','Covariate $\ddag$')
    df_agg.loc[:,'vars'] = df_agg.loc[:, 'vars'].replace('Y','Outcome $\ddag$')
    df_agg_pivot = df_agg.pivot(index=['decomp'], columns='vars', values='decision')
    df_agg_annot_pivot = df_agg.pivot(index=['decomp'], columns='vars', values='pvalue')
    df_agg_pivot = df_agg_pivot.reset_index(drop=True)
    df_agg_annot_pivot = df_agg_annot_pivot.reset_index(drop=True)
    df_agg_pivot.columns.name = ''
    print(df_agg_pivot.columns)
    print(df_agg_annot_pivot.columns)
    
    plt.figure(figsize=(5,1))
    ax = sns.heatmap(1 - df_agg_pivot, 
                    annot=1 - df_agg_annot_pivot, fmt='.2f', 
                    cmap=cmap, vmin=0, vmax=1,
                    yticklabels=False, linewidths=2,
                    #  cbar_kws={'label': 'pvalue'},
                    cbar = False,
        )
    ax.xaxis.tick_top()
    ax.set_title('Aggregate', pad=10)
    plt.savefig(result_data.replace('.csv','target_inference_agg.pdf'), dpi=300, bbox_inches='tight')
    plt.close()

    return df_agg_annot_pivot, df_agg_pivot

def get_group_names(grouping_data):
    names = pd.read_csv(grouping_data)
    names.loc[:, 'var'] = 'X' + (names.loc[:, 'var_idx']+1).astype(str)
    names['group_vars'] = names.groupby('group_name')['var'].transform(lambda x: ",".join(x))
    names = names[['group_vars', 'group_name']].drop_duplicates()
    return names

def plot_detailed(result_data, decomp, names):
    df = pd.read_csv(result_data)
    df_detail = df[(df.decomp == decomp) & (df.est == 'onestep')]
    df_detail.loc[:, 'vars'] = df_detail.loc[:, 'vars'].apply(parse_proposed_vars)
    df_detail_annot = df_detail[['vars','pvalue']]
    df_detail = df_detail[['vars','decision']]
    def pivot_trim_df(df_detail):
        if names is not None:
            df_detail = df_detail.merge(names, left_on="vars", right_on="group_vars")
            df_detail = df_detail.drop(['vars','group_vars'], axis=1)
            df_detail.set_index('group_name', inplace=True)
            df_detail = df_detail.sort_values(by='group_name')
        else:
            df_detail.set_index('vars', inplace=True)
            df_detail = df_detail.sort_values(by='vars')
        df_detail.index.name = None
        return df_detail
    df_detail_annot = pivot_trim_df(df_detail_annot)
    df_detail = pivot_trim_df(df_detail)
    # df_detail.columns = ['SHIFT']
    print(df_detail.index)
    print(df_detail_annot.index)

    plt.figure(figsize=(2,3))
    ax = sns.heatmap(df_detail, 
                    annot=df_detail_annot, fmt='.2f', 
                    cmap=cmap, vmin=0, vmax=1,
                    xticklabels=False, 
                    yticklabels=False,
                    cbar=False
                    )
    ax.tick_params(axis='y', rotation=0)
    ax.set_title('Detailed $\dagger$')
    plt.savefig(result_data.replace('.csv', '_detail.pdf'), dpi=300, bbox_inches='tight')
    plt.close()

    return df_detail_annot, df_detail

def plot_comparators_real(comp_data, target_data, decomp, names):
    num_p = pd.read_csv(target_data).shape[1] - 1  # number of features
    
    df = pd.read_csv(comp_data)

    # Remove rows for CausalForestExplanation
    df = df[df['est'] != 'CausalForestExplanation']

    # Remove rows for DomainClassifierExplanation
    df = df[df['est'] != 'DomainClassifierExplanation']

    # Do Bonferroni correction on each est group
    df['significance_level'] = SIGNIFICANCE_LEVEL
    if BONFERRONI_CORRECTION:
        df['significance_level'] = df.groupby(['est'])['significance_level'].transform(lambda x: x / len(x))

    # Get excluded vars for TEVIM and KCI tests, as the vars column logs complement of the variable of interest
    explanation_tests = df.est.str.contains('KCIOutcomeTest|TEVIMTest|KCICovariateTest')
    df.loc[explanation_tests, 'vars'] = df.loc[explanation_tests, 'vars'].transform(lambda x: parse_comp_vars(x, num_p))

    # Decide test by comparing to significance level
    df.loc[:, 'significant'] = False
    df.loc[explanation_tests, 'significant'] = np.where(df.loc[explanation_tests, 'pvalue'] < df.loc[explanation_tests, 'significance_level'], False, True)
    df.loc[~explanation_tests, 'significant'] = np.where(df.loc[~explanation_tests, 'pvalue'] < df.loc[~explanation_tests, 'significance_level'], True, False)
    df['significant'] = (df.loc[:, 'significant']).astype(float)

    # Keep explanation tests to plot
    df = df.loc[explanation_tests]

    df = df.merge(names, left_on='vars', right_on='group_vars', how='left')

    # Keep individual var names if no group found
    df.loc[df['group_vars'].isna(), 'group_name'] = df.loc[df['group_vars'].isna(), 'vars']

    # Pivot df on vars and est columns
    df_pivot = df.pivot(index=['group_name'], columns='est', values='significant')
    df_annot_pivot = df.pivot(index=['group_name'], columns='est', values='pvalue')

    # Sort by vars
    df_pivot = df_pivot.reset_index()
    df_annot_pivot = df_annot_pivot.reset_index()

    df_pivot.set_index('group_name', inplace=True)
    df_annot_pivot.set_index('group_name', inplace=True)

    # Write to csv
    df_annot_pivot.to_csv(comp_data.replace('.csv', '_pivot.csv'), index=False)
    print(df_pivot.columns)
    print(df_annot_pivot.columns)

    explanation_tests_cols = df_pivot.columns.str.contains('KCIOutcomeTest|TEVIMTest|KCICovariateTest')
    df_pivot.loc[:, explanation_tests_cols] = 1 - df_pivot.loc[:, explanation_tests_cols]
    df_pivot.columns.name = None
    df_pivot.index.name = None
    if df_pivot.shape[1] == 1:
        plt.figure(figsize=(2,3))
    else:
        plt.figure(figsize=(4,3))
    df_pivot = df_pivot.rename(columns=COMPARATOR_NAMES)
    ax = sns.heatmap(df_pivot, 
                    annot=df_annot_pivot, fmt='.2f',
                    cmap=cmap, vmin=0, vmax=1,
                    yticklabels=decomp=='Cond_Cov',
                    cbar=False,
                    )
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)
    ax.set_title('Baselines %s' % COMPARATOR_NAMES[decomp])
    plt.savefig(comp_data.replace('.csv', '_detail.pdf'), dpi=300, bbox_inches='tight')
    plt.close()

    return df_annot_pivot, df_pivot

In [33]:
for simulation in SIMULATION_SETTINGS:
    print('simulation: %s\n' % simulation)
    if simulation == 'readmission_zsfg_ucsf_fixed_covariate':
        result_data = SIMULATION_SETTINGS[simulation]['result_data']
        decomp = SIMULATION_SETTINGS[simulation]['decomp']
        grouping_data = SIMULATION_SETTINGS[simulation]['grouping_data']
        comp_data = SIMULATION_SETTINGS[simulation]['comp_data']
        target_data = SIMULATION_SETTINGS[simulation]['target_data']
        
        # Get group names
        names = get_group_names(grouping_data)
        
        plot_aggregate(result_data)
        plot_detailed(result_data, decomp, names)
        plot_comparators_real(comp_data, target_data, decomp, names)
        break

simulation: shield_nonzero_covariate

simulation: acs_pubcov_outcome

simulation: acs_pubcov_covariate

simulation: readmission_ucsf_zsfg_covariate

simulation: readmission_ucsf_zsfg_outcome

simulation: readmission_zsfg_ucsf_fixed_covariate

Index(['Covariate $\ddag$', 'Outcome $\ddag$'], dtype='object', name='')
Index(['Covariate $\ddag$', 'Outcome $\ddag$'], dtype='object', name='vars')
Index(['Demo', 'Encounters', 'Labs', 'Vitals'], dtype='object')
Index(['Demo', 'Encounters', 'Labs', 'Vitals'], dtype='object')
Index(['KCICovariateTest'], dtype='object', name='est')
Index(['KCICovariateTest'], dtype='object', name='est')


In [34]:
0.849206 - 0.644404

0.20480200000000004

In [1]:
0.845336 - 0.687568

0.15776800000000002

In [3]:
0.94557 - 0.826947

0.11862300000000003

# Simulations

In [46]:
NUM_JOBS = 50
SIMULATION_SETTINGS = {
    'dim4_linear_std': {
        # 'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/dim4_linear_std/resultestimate1.csv',
        # 'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/dim4_linear_std/comp_resultestimate1.csv',
        'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_comparators/_output/dim4_linear_std_save/accuracy/0.05/40/8000/resultestimateJOB.csv',
        'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_comparators/_output/dim4_linear_std_save/accuracy/0.05/40/8000/comp_resultestimateJOB.csv',
        'grouping_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation/feature_grouping_dim4_norm1.csv',
        'names_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation/feature_names_dim4_norm1.csv',
        'target_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_comparators/_output/dim4_linear_std_save/8000/target_data1.csv',
        'decomp': "Cond_Outcome",
    },
    'dim4_norm2_std': {
        # 'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/dim4_norm2_stdshift/resultestimate1.csv',
        # 'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/remote_result/explain_score_test/dim4_norm2_stdshift/res_comparators.csv',
        'result_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_comparators/_output/dim4_norm2_stdshiftfirstvar/accuracy/0.02/40/8000/resultestimateJOB.csv',
        'comp_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_comparators/_output/dim4_norm2_stdshiftfirstvar/accuracy/0.02/40/8000/comp_resultestimateJOB.csv',
        'grouping_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation/feature_grouping_dim4_norm1.csv',
        'names_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation/feature_names_dim4_norm1.csv',
        'target_data': '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_comparators/_output/dim4_norm2_stdshiftfirstvar/8000/target_data1.csv',
        'decomp': "Cond_Cov",
    }
}

In [71]:
def concat_files_to_df(files, num_jobs):
    all_res = []
    for file in files:
        if num_jobs is not None:
            for job_idx in range(num_jobs):
                file_jobidx = file.replace("JOB", str(job_idx+1))
                if os.path.exists(file_jobidx):
                    all_res.append(pd.read_csv(file_jobidx, delimiter=',', quotechar='"'))
                else:
                    print("FILE MISSING", file_jobidx)
        else:
            all_res.append(pd.read_csv(file, delimiter=',', quotechar='"'))
    return pd.concat(all_res)

def plot_aggregate_simulation(df):
    df_agg = df[(df.decomp == 'agg') & (df.est == 'onestep')]
    df_agg.loc[:,'vars'] = df_agg.loc[:, 'vars'].replace('X','Covariate $\ddag$')
    df_agg.loc[:,'vars'] = df_agg.loc[:, 'vars'].replace('Y','Outcome $\ddag$')
    df_agg = df_agg.groupby(by=['level','decomp','vars','est','mdl','nsource','ntarget'])['pvalue'].median()
    print(df_agg)
    df_agg = df_agg.reset_index()
    df_agg_pivot = df_agg.pivot(index=['decomp'], columns='vars', values='pvalue') <= SIGNIFICANCE_LEVEL
    df_agg_pivot = df_agg_pivot.reset_index(drop=True)
    df_agg_pivot.columns.name = ''
    print(df_agg_pivot.columns)
    print(df_agg_pivot)
    
    plt.figure(figsize=(5,1))
    ax = sns.heatmap(~df_agg_pivot, 
                    annot=df_agg_pivot, fmt='.2f', 
                    cmap=cmap, vmin=0, vmax=1,
                    yticklabels=False, linewidths=2,
                    #  cbar_kws={'label': 'pvalue'},
                    cbar = False,
        )
    ax.xaxis.tick_top()
    ax.set_title('Aggregate', pad=10)
    plt.savefig(result_data.replace('.csv','target_inference_agg.pdf'), dpi=300, bbox_inches='tight')
    plt.close()

    return df_agg_pivot

def plot_detailed_simulation(df, decomp):
    df_detail = df[(df.decomp == decomp) & (df.est == 'onestep')]
    df_detail.loc[:, 'vars'] = df_detail.loc[:, 'vars'].apply(parse_proposed_vars)
    df_detail = df_detail[~df_detail.vars.str.contains('X1,X4|X2,X3|X1,X2|X3,X4')]
    df_detail = df_detail.groupby(by=['level','decomp','vars','est','mdl','nsource','ntarget'])['pvalue'].median()
    df_detail = df_detail.reset_index()
    df_detail = df_detail[['vars','pvalue']]
    df_detail.set_index('vars', inplace=True)
    df_detail.index.name = None
    # df_detail.columns = ['SHIFT']
    print(df_detail.index)
    print(df_detail)

    plt.figure(figsize=(2,3))
    ax = sns.heatmap(df_detail <= SIGNIFICANCE_LEVEL, 
                    annot=df_detail, fmt='.2f', 
                    cmap=cmap, vmin=0, vmax=1,
                    xticklabels=False,
                    yticklabels=decomp=='Cond_Outcome',
                    cbar=False
                    )
    ax.tick_params(axis='y', rotation=0)
    ax.set_title('Detailed $\dagger$')
    plt.savefig(result_data.replace('.csv', '_detail.pdf'), dpi=300, bbox_inches='tight')
    plt.close()

    return df_detail

def plot_comparators_simulation(comp_data, target_data, decomp):
    num_p = pd.read_csv(target_data).shape[1] - 1  # number of features
    
    df = concat_files_to_df([comp_data], NUM_JOBS)
    print(df)
    
    # Remove rows for CausalForestExplanation
    df = df[df['est'] != 'CausalForestExplanation']

    # Remove rows for DomainClassifierExplanation
    df = df[df['est'] != 'DomainClassifierExplanation']

    # Do Bonferroni correction on each est group
    df['significance_level'] = SIGNIFICANCE_LEVEL
    if BONFERRONI_CORRECTION:
        df['significance_level'] = df.groupby(['nsource','ntarget','est'])['significance_level'].transform(lambda x: x / len(x))

    # Get excluded vars for TEVIM and KCI tests, as the vars column logs complement of the variable of interest
    explanation_tests = df.est.str.contains('KCIOutcomeTest|TEVIMTest|KCICovariateTest')
    df.loc[explanation_tests, 'vars'] = df.loc[explanation_tests, 'vars'].transform(lambda x: parse_comp_vars(x, num_p))
    df = df[~df.vars.str.contains('X1,X4|X2,X3|X1,X2|X3,X4')]

    # Decide test by comparing to significance level
    # df_annot = df
    # df.loc[:,'significant'] = df.loc[:,'pvalue'] < df.loc[:,'significance_level']
    # df['significant'] = df['significant'].astype(float)
    df = df.groupby(by=['level','decomp','vars','est','mdl','nsource','ntarget'])['pvalue'].median()
    df = df.reset_index()

    # Pivot df on vars and est columns
    df_pivot = df.pivot(index=['vars','nsource','ntarget'], columns='est', values='pvalue')

    def trim_dataframe(df_pivot):
        # Sort by number of vars
        df_pivot = df_pivot.reset_index()
        df_pivot['vars_num'] = df_pivot['vars'].transform(lambda x: x.count(','))

        # Merge by var_idx in names and sort by vars_index
        df_pivot = df_pivot.sort_values(by=['nsource','vars_num'], ascending=True)

        df_pivot = df_pivot[df_pivot.nsource == 8000]

        # Drop all indices, group name, and vars columns
        df_pivot = df_pivot.drop(columns=['vars_num', 'nsource', 'ntarget'])  # assumes nsource == ntarget
        df_pivot.set_index('vars', inplace=True)
        df_pivot.index.name = None
        df_pivot.columns.name = None
        return df_pivot

    df_pivot = trim_dataframe(df_pivot)

    # Write to csv
    df_pivot.to_csv(comp_data.replace('.csv', '_pivot.csv'), index=False)
    print(df_pivot.index)

    explanation_tests_cols = df_pivot.columns.str.contains('KCIOutcomeTest|TEVIMTest|KCICovariateTest')  # fail to reject means variables are flagged
    df_pivot.loc[:, explanation_tests_cols] = 1 - df_pivot.loc[:, explanation_tests_cols]  # fail-to-reject rate

    df_pivot = df_pivot.rename(columns=COMPARATOR_NAMES)
    plt.figure(figsize=(9,3))
    ax = sns.heatmap(df_pivot > SIGNIFICANCE_LEVEL, 
                    annot=1 - df_pivot, fmt='.2f',
                    cmap=cmap, vmin=0, vmax=1,
                    yticklabels=decomp=='Cond_Cov',
                    cbar=False
                    )
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)
    ax.set_title('Baselines %s' % COMPARATOR_NAMES[decomp])
    plt.savefig(comp_data.replace('.csv', '_detail.pdf'), dpi=300, bbox_inches='tight')
    plt.close()

    return df_pivot

In [72]:
for simulation in SIMULATION_SETTINGS:
    print('simulation: %s\n' % simulation)
    result_data = SIMULATION_SETTINGS[simulation]['result_data']
    decomp = SIMULATION_SETTINGS[simulation]['decomp']
    comp_data = SIMULATION_SETTINGS[simulation]['comp_data']
    target_data = SIMULATION_SETTINGS[simulation]['target_data']

    result_df = concat_files_to_df([result_data], NUM_JOBS)
    print(result_df)
    plot_aggregate_simulation(result_df)
    plot_detailed_simulation(result_df, decomp)
    df = plot_comparators_simulation(comp_data, target_data, decomp)
    # break

simulation: dim4_linear_std

       value  critical_value  decision  pvalue   level        decomp  \
0   0.000000        0.000000     False  1.0000     agg           agg   
1   0.057618        0.016299      True  0.0000     agg           agg   
2   0.000000        0.000000     False  1.0000     agg           agg   
3   0.031150        0.010300      True  0.0000     agg           agg   
4  -0.019280        0.037681     False  0.8041  detail  Cond_Outcome   
..       ...             ...       ...     ...     ...           ...   
9   0.058673        0.033640      True  0.0015  detail  Cond_Outcome   
10  0.135372        0.019605      True  0.0000  detail  Cond_Outcome   
11  0.097397        0.020479      True  0.0000  detail  Cond_Outcome   
12  0.129438        0.019152      True  0.0000  detail  Cond_Outcome   
13  0.000000        0.000000     False  1.0000  detail  Cond_Outcome   

                          vars      est  job  \
0                            X   plugin    1   
1         